# 06 - Simple ML Forecast Baselines

In this notebook, we begin the machine learning side of the project with simple, interpretable forecast baselines.

The first decision is what kind of prediction problem we are building:

- Same-month prediction
- True one-step-ahead forecasting

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()

PROJECT_ROOT

In [ ]:
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "martin_selected_30_monthly_production_normalized.csv"

DATA_FILE

In [ ]:
DATA_FILE.exists()

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

PROJECT_ROOT

In [ ]:
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "martin_selected_30_monthly_production_normalized.csv"

DATA_FILE

In [ ]:
df = pd.read_csv(DATA_FILE)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
id_columns = [
    "api8",
    "district",
    "lease_no",
    "well_no",
]

date_label_columns = [
    "first_prod_month",
    "cycle_year_month",
    "reported_first_month",
    "first_positive_prod_month",
]

numeric_columns = [
    "month_on_production",
    "oil_bbl",
    "casinghead_gas_mcf",
    "boe",
    "interval_length_proxy_ft",
    "reported_month_on_production",
]

for column in id_columns:
    df[column] = df[column].astype(str)

for column in date_label_columns:
    df[column] = df[column].astype("int64").astype(str)

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.dtypes

In [ ]:
numeric_columns = [
    "month_on_production",
    "oil_bbl",
    "casinghead_gas_mcf",
    "boe",
    "interval_length_proxy_ft",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

In [ ]:
df = df.sort_values(["api8", "month_on_production"]).reset_index(drop=True)

df.head()

In [ ]:
df[["month_on_production", "oil_bbl", "interval_length_proxy_ft"]].describe()

In [ ]:
df["api8"].nunique()

In [ ]:
modeling_df = df[df["month_on_production"].between(1, 33)].copy()

modeling_df.shape


In [ ]:
well_groups = modeling_df.groupby("api8", group_keys=False)


In [ ]:
modeling_df["target_month_on_production"] = well_groups["month_on_production"].shift(-1)

modeling_df[["api8", "month_on_production", "target_month_on_production"]].head(10)

In [ ]:
modeling_df["target_next_oil_bbl"] = well_groups["oil_bbl"].shift(-1)

modeling_df[
    [
        "api8",
        "month_on_production",
        "oil_bbl",
        "target_month_on_production",
        "target_next_oil_bbl",
    ]
].head(10)

In [ ]:
modeling_df["last_observed_oil_bbl"] = modeling_df["oil_bbl"]

modeling_df[
    [
        "api8",
        "month_on_production",
        "oil_bbl",
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
    ]
].head(10)

In [ ]:
modeling_df["trailing_3mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)

modeling_df[
    [
        "api8",
        "month_on_production",
        "oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "target_next_oil_bbl",
    ]
].head(10)

In [ ]:
modeling_df["trailing_6mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

modeling_df[
    [
        "api8",
        "month_on_production",
        "oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
        "target_next_oil_bbl",
    ]
].head(10)

In [ ]:
one_step_rows = modeling_df.dropna(
    subset=[
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
    ]
).copy()

one_step_rows.shape

In [ ]:
one_step_rows[
    [
        "api8",
        "month_on_production",
        "oil_bbl",
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
        "interval_length_proxy_ft",
    ]
].head(10)

In [ ]:
train_rows = one_step_rows[one_step_rows["target_month_on_production"].between(13, 24)].copy()

forecast_check_rows = one_step_rows[one_step_rows["target_month_on_production"].between(25, 33)].copy()

train_rows.shape, forecast_check_rows.shape

In [ ]:
forecast_check_rows["naive_last_observed_forecast_oil_bbl"] = forecast_check_rows["last_observed_oil_bbl"]

forecast_check_rows[
    [
        "api8",
        "month_on_production",
        "target_month_on_production",
        "last_observed_oil_bbl",
        "target_next_oil_bbl",
        "naive_last_observed_forecast_oil_bbl",
    ]
].head(10)

In [ ]:
forecast_check_rows["naive_error_bbl"] = (
    forecast_check_rows["naive_last_observed_forecast_oil_bbl"]
    - forecast_check_rows["target_next_oil_bbl"]
)

forecast_check_rows["naive_absolute_error_bbl"] = forecast_check_rows["naive_error_bbl"].abs()

forecast_check_rows[
    [
        "api8",
        "target_month_on_production",
        "target_next_oil_bbl",
        "naive_last_observed_forecast_oil_bbl",
        "naive_error_bbl",
        "naive_absolute_error_bbl",
    ]
].head(10)

In [ ]:
naive_mae_bbl = forecast_check_rows["naive_absolute_error_bbl"].mean()
naive_bias_bbl = forecast_check_rows["naive_error_bbl"].mean()

naive_mae_bbl, naive_bias_bbl

In [ ]:
baseline_metrics = pd.DataFrame(
    [
        {
            "model": "naive_last_observed",
            "mae_bbl": naive_mae_bbl,
            "bias_bbl": naive_bias_bbl,
        }
    ]
)

baseline_metrics

In [ ]:
forecast_check_rows["trailing_3mo_avg_forecast_oil_bbl"] = forecast_check_rows[
    "trailing_3mo_avg_oil_bbl"
]

forecast_check_rows[
    [
        "api8",
        "target_month_on_production",
        "target_next_oil_bbl",
        "trailing_3mo_avg_forecast_oil_bbl",
    ]
].head(10)

In [ ]:
forecast_check_rows["trailing_3mo_error_bbl"] = (
    forecast_check_rows["trailing_3mo_avg_forecast_oil_bbl"]
    - forecast_check_rows["target_next_oil_bbl"]
)

forecast_check_rows["trailing_3mo_absolute_error_bbl"] = forecast_check_rows[
    "trailing_3mo_error_bbl"
].abs()

forecast_check_rows[
    [
        "api8",
        "target_month_on_production",
        "target_next_oil_bbl",
        "trailing_3mo_avg_forecast_oil_bbl",
        "trailing_3mo_error_bbl",
        "trailing_3mo_absolute_error_bbl",
    ]
].head(10)

In [ ]:
trailing_3mo_mae_bbl = forecast_check_rows["trailing_3mo_absolute_error_bbl"].mean()
trailing_3mo_bias_bbl = forecast_check_rows["trailing_3mo_error_bbl"].mean()

trailing_3mo_mae_bbl, trailing_3mo_bias_bbl

In [ ]:
baseline_metrics = pd.concat(
    [
        baseline_metrics,
        pd.DataFrame(
            [
                {
                    "model": "trailing_3mo_average",
                    "mae_bbl": trailing_3mo_mae_bbl,
                    "bias_bbl": trailing_3mo_bias_bbl,
                }
            ]
        ),
    ],
    ignore_index=True,
)

baseline_metrics

In [ ]:
forecast_check_rows["trailing_6mo_avg_forecast_oil_bbl"] = forecast_check_rows[
    "trailing_6mo_avg_oil_bbl"
]

forecast_check_rows["trailing_6mo_error_bbl"] = (
    forecast_check_rows["trailing_6mo_avg_forecast_oil_bbl"]
    - forecast_check_rows["target_next_oil_bbl"]
)

forecast_check_rows["trailing_6mo_absolute_error_bbl"] = forecast_check_rows[
    "trailing_6mo_error_bbl"
].abs()

trailing_6mo_mae_bbl = forecast_check_rows["trailing_6mo_absolute_error_bbl"].mean()
trailing_6mo_bias_bbl = forecast_check_rows["trailing_6mo_error_bbl"].mean()

trailing_6mo_mae_bbl, trailing_6mo_bias_bbl

In [ ]:
baseline_metrics = pd.concat(
    [
        baseline_metrics,
        pd.DataFrame(
            [
                {
                    "model": "trailing_6mo_average",
                    "mae_bbl": trailing_6mo_mae_bbl,
                    "bias_bbl": trailing_6mo_bias_bbl,
                }
            ]
        ),
    ],
    ignore_index=True,
)

baseline_metrics

In [ ]:
naive_wape = (
    forecast_check_rows["naive_absolute_error_bbl"].sum()
    / forecast_check_rows["target_next_oil_bbl"].sum()
)

trailing_3mo_wape = (
    forecast_check_rows["trailing_3mo_absolute_error_bbl"].sum()
    / forecast_check_rows["target_next_oil_bbl"].sum()
)

trailing_6mo_wape = (
    forecast_check_rows["trailing_6mo_absolute_error_bbl"].sum()
    / forecast_check_rows["target_next_oil_bbl"].sum()
)

naive_wape, trailing_3mo_wape, trailing_6mo_wape

In [ ]:
baseline_metrics["wape"] = [
    naive_wape,
    trailing_3mo_wape,
    trailing_6mo_wape,
]

baseline_metrics.sort_values("mae_bbl")

In [ ]:
baseline_metrics_display = baseline_metrics.copy()

baseline_metrics_display["wape_percent"] = baseline_metrics_display["wape"] * 100

baseline_metrics_display.sort_values("mae_bbl")

In [ ]:
baseline_metrics_display.sort_values("mae_bbl").round(2)

## First Baseline Results

The simple baseline with the lowest mean absolute error is 'naive_last_observed'.

Its average absolute error is approximately '699.33' barrels per forecasted month.

Its WAPE is approximately 19.50%.

The bias is 91.51, which means this baseline tends to OVER_FORECAST oil production over the forecast-check rows.

These are rolling one-step-ahead baselines, not fixed-origin forecasts from month 24. That means they are useful first benchmarks, but we should be careful before comparing them directly against the DCA forecast totals.

In [ ]:
import matplotlib.pyplot as plt

best_baseline_name = (
    baseline_metrics
    .sort_values("mae_bbl")
    .iloc[0]["model"]
)

best_baseline_name

In [ ]:
baseline_forecast_columns = {
    "naive_last_observed": "naive_last_observed_forecast_oil_bbl",
    "trailing_3mo_average": "trailing_3mo_avg_forecast_oil_bbl",
    "trailing_6mo_average": "trailing_6mo_avg_forecast_oil_bbl",
}

best_forecast_column = baseline_forecast_columns[best_baseline_name]

best_forecast_column

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    forecast_check_rows["target_next_oil_bbl"],
    forecast_check_rows[best_forecast_column],
    alpha=0.7,
)

plt.plot(
    [
        forecast_check_rows["target_next_oil_bbl"].min(),
        forecast_check_rows["target_next_oil_bbl"].max(),
    ],
    [
        forecast_check_rows["target_next_oil_bbl"].min(),
        forecast_check_rows["target_next_oil_bbl"].max(),
    ],
    linestyle="--",
)

plt.title(f"Actual vs Forecast Oil: {best_baseline_name}")
plt.xlabel("Actual next-month oil (bbl)")
plt.ylabel("Forecast next-month oil (bbl)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Actual vs Forecast Check

The scatter plot compares actual next-month oil production against the forecast from the best simple baseline.

Points near the dashed diagonal line are more accurate forecasts.

Points above the line are over-forecasts.

Points below the line are under-forecasts.

This plot helps show whether the baseline errors are evenly spread or whether the model struggles more at high-production or low-production months.

## Simple ML Feature Table

Now that we have simple non-ML baselines, we can prepare a feature table for basic machine learning models.

The target is still next-month oil production.

The features should only use information available at the current forecast month.

For the first simple feature set, we will use:

- Current month on production
- Last observed oil production
- Trailing 3-month average oil production
- Trailing 6-month average oil production
- Interval length proxy

## Simple ML Methods

The simple forecasting baselines used fixed rules.

Now we will let a model learn a relationship between input features and next-month oil production.

The target is:

- `target_next_oil_bbl`

The first feature set will use only information available at forecast time:

- `month_on_production`
- `last_observed_oil_bbl`
- `trailing_3mo_avg_oil_bbl`
- `trailing_6mo_avg_oil_bbl`
- `interval_length_proxy_ft`

This keeps the ML setup leakage-safe for one-step-ahead forecasting.

In [ ]:
feature_columns = [
    "month_on_production",
    "last_observed_oil_bbl",
    "trailing_3mo_avg_oil_bbl",
    "trailing_6mo_avg_oil_bbl",
    "interval_length_proxy_ft",
]

target_column = "target_next_oil_bbl"

feature_columns, target_column

In [ ]:
X_train = train_rows[feature_columns]
y_train = train_rows[target_column]

X_train.head()

In [ ]:
X_check = forecast_check_rows[feature_columns]
y_check = forecast_check_rows[target_column]

X_check.head()


In [ ]:
X_train.isna().sum()

In [ ]:
X_check.isna().sum()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

In [ ]:
forecast_check_rows["linear_regression_forecast_oil_bbl"] = linear_model.predict(X_check)

forecast_check_rows[
    [
        "api8",
        "target_month_on_production",
        "target_next_oil_bbl",
        "linear_regression_forecast_oil_bbl",
    ]
].head(10)

In [ ]:
df.loc[df["api8"].astype(str) == "31743549", [
    "api8",
    "month_on_production",
    "oil_bbl",
    "cycle_year_month",
    "lease_name",
    "well_no",
]]

In [ ]:
forecast_check_rows["linear_regression_error_bbl"] = (
    forecast_check_rows["linear_regression_forecast_oil_bbl"]
    - forecast_check_rows["target_next_oil_bbl"]
)

forecast_check_rows["linear_regression_absolute_error_bbl"] = forecast_check_rows[
    "linear_regression_error_bbl"
].abs()

forecast_check_rows[
    [
        "api8",
        "target_month_on_production",
        "target_next_oil_bbl",
        "linear_regression_forecast_oil_bbl",
        "linear_regression_error_bbl",
        "linear_regression_absolute_error_bbl",
    ]
].head(10)

In [ ]:
linear_regression_mae_bbl = forecast_check_rows[
    "linear_regression_absolute_error_bbl"
].mean()

linear_regression_bias_bbl = forecast_check_rows[
    "linear_regression_error_bbl"
].mean()

linear_regression_wape = (
    forecast_check_rows["linear_regression_absolute_error_bbl"].sum()
    / forecast_check_rows["target_next_oil_bbl"].sum()
)

linear_regression_mae_bbl, linear_regression_bias_bbl, linear_regression_wape

In [ ]:
baseline_metrics = pd.concat(
    [
        baseline_metrics,
        pd.DataFrame(
            [
                {
                    "model": "linear_regression",
                    "mae_bbl": linear_regression_mae_bbl,
                    "bias_bbl": linear_regression_bias_bbl,
                    "wape": linear_regression_wape,
                }
            ]
        ),
    ],
    ignore_index=True,
)

baseline_metrics.sort_values("mae_bbl").round(2)

In [ ]:
linear_coefficients = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": linear_model.coef_,
    }
)

linear_coefficients

In [ ]:
linear_model.intercept_

## Linear Regression Coefficients

The linear regression coefficients mostly follow the expected direction.

`month_on_production` has a negative coefficient, which is consistent with production decline as wells age.

`last_observed_oil_bbl`, `trailing_3mo_avg_oil_bbl`, and `trailing_6mo_avg_oil_bbl` all have positive coefficients, meaning stronger recent production history increases the next-month forecast.

The `interval_length_proxy_ft` coefficient is negative in this first model. This should not be interpreted as a strong geologic conclusion. It may reflect feature scaling, correlation with other variables, or patterns specific to this small 30-well cohort.

Because the features are measured on different scales and are correlated with each other, these coefficients are best treated as a first diagnostic rather than final feature importance.

## Random Forest Baseline

Next we test a random forest model.

A random forest can learn non-linear relationships between the features and next-month oil production.

Unlike linear regression, it is not limited to one straight-line formula.

However, it is also less transparent, so we will still compare it using the same forecast-check metrics:

- MAE
- bias
- WAPE

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
random_forest_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=4,
    random_state=42,
)

random_forest_model.fit(X_train, y_train)

In [ ]:
forecast_check_rows["random_forest_forecast_oil_bbl"] = random_forest_model.predict(X_check)

forecast_check_rows[
    [
        "api8",
        "target_month_on_production",
        "target_next_oil_bbl",
        "random_forest_forecast_oil_bbl",
    ]
].head(10)


In [ ]:
forecast_check_rows["random_forest_error_bbl"] = (
    forecast_check_rows["random_forest_forecast_oil_bbl"]
    - forecast_check_rows["target_next_oil_bbl"]
)

forecast_check_rows["random_forest_absolute_error_bbl"] = forecast_check_rows[
    "random_forest_error_bbl"
].abs()

forecast_check_rows[
    [
        "api8",
        "target_month_on_production",
        "target_next_oil_bbl",
        "random_forest_forecast_oil_bbl",
        "random_forest_error_bbl",
        "random_forest_absolute_error_bbl",
    ]
].head(10)


In [ ]:
random_forest_mae_bbl = forecast_check_rows[
    "random_forest_absolute_error_bbl"
].mean()

random_forest_bias_bbl = forecast_check_rows[
    "random_forest_error_bbl"
].mean()

random_forest_wape = (
    forecast_check_rows["random_forest_absolute_error_bbl"].sum()
    / forecast_check_rows["target_next_oil_bbl"].sum()
)

random_forest_mae_bbl, random_forest_bias_bbl, random_forest_wape

In [ ]:
baseline_metrics = pd.concat(
    [
        baseline_metrics,
        pd.DataFrame(
            [
                {
                    "model": "random_forest",
                    "mae_bbl": random_forest_mae_bbl,
                    "bias_bbl": random_forest_bias_bbl,
                    "wape": random_forest_wape,
                }
            ]
        ),
    ],
    ignore_index=True,
)

baseline_metrics.sort_values("mae_bbl").round(2)

In [ ]:
random_forest_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": random_forest_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

random_forest_importance

## Random Forest Result

The random forest is the second ML model in this notebook.

It can learn non-linear patterns, but it still uses the same leakage-safe one-step-ahead feature table as linear regression.

The random forest should be compared against both:

- the simple forecasting baselines
- the linear regression baseline

The feature importance table shows which inputs the random forest relied on most.

As with the linear regression coefficients, these importances should be treated as model diagnostics, not causal explanations.

The key question is whether the random forest improves on the naive last-observed forecast. If it does not, then the simple carry-forward rule remains the strongest baseline so far.

In [ ]:
final_metrics_display = baseline_metrics.copy()

final_metrics_display["wape_percent"] = final_metrics_display["wape"] * 100

final_metrics_display = final_metrics_display.sort_values("mae_bbl").reset_index(drop=True)

final_metrics_display.round(2)

In [ ]:
ML_OUTPUT_DIR = PROJECT_ROOT / "reports" / "ml_outputs"

ML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_metrics_display.to_csv(
    ML_OUTPUT_DIR / "simple_ml_baseline_metrics.csv",
    index=False,
)

In [ ]:
forecast_output_columns = [
    "api8",
    "lease_name",
    "well_no",
    "month_on_production",
    "target_month_on_production",
    "target_next_oil_bbl",
    "naive_last_observed_forecast_oil_bbl",
    "trailing_3mo_avg_forecast_oil_bbl",
    "trailing_6mo_avg_forecast_oil_bbl",
    "linear_regression_forecast_oil_bbl",
    "random_forest_forecast_oil_bbl",
]

forecast_check_rows[forecast_output_columns].to_csv(
    ML_OUTPUT_DIR / "simple_ml_forecast_check_predictions.csv",
    index=False,
)

## Notebook Conclusion

This notebook built the first simple machine learning forecast baselines for the Martin County 30-well cohort.

The forecast target was next-month oil production:

- use information available through the current month
- predict oil production in the next month

The notebook first tested simple rule-based forecasting baselines:

- naive last-observed oil
- trailing 3-month average oil
- trailing 6-month average oil

Then it tested simple ML models:

- linear regression
- random forest

All methods were evaluated on target months 25-33 using the same metrics:

- MAE
- bias
- WAPE

The best method in this rolling one-step-ahead setup was 'naive_last_observed'.

This result is important because any future ML model should be compared against the naive last-observed baseline, not just against more complex models.

Important limitation: this is a rolling one-step-ahead evaluation. It is not yet the same as the fixed-origin DCA setup, where forecasts for months 25-33 are made from information available only through month 24.

## Next Steps

The next modeling step is to make the ML comparison stricter and more directly comparable to DCA.

Possible next steps:

- Build a fixed-origin ML forecast that uses information through month 24 only
- Forecast months 25-33 without updating features with observed forecast-check months
- Compare fixed-origin ML totals against the DCA per-well forecast totals
- Test whether additional leakage-safe features improve performance
- Keep gas forecasting out of release 1

## QA Checklist

Before using these results in the project summary:

- Confirm the notebook runs from top to bottom without errors
- Confirm `api8`, `lease_no`, and date-label columns are treated as labels, not ML numeric features
- Confirm the ML feature list excludes same-month or future target information
- Confirm the forecast-check window uses target months 25-33
- Confirm the final metrics table and prediction CSV files were saved under `reports/ml_outputs/`
- Remember that this notebook is rolling one-step-ahead, not fixed-origin DCA-style forecasting

In [ ]:
sorted(path.name for path in ML_OUTPUT_DIR.glob("*.csv"))